# ASEAN Top-100 — chạy thực nghiệm trên Google Colab

Notebook này chạy lại pipeline trên bộ Top-100 availability-aware đã đóng gói cho Indonesia, Malaysia, Philippines, Singapore và Thailand. Dữ liệu phải được upload lên Google Drive trước; notebook không tải lại dữ liệu LSEG.

Mỗi nước được chạy riêng để không trộn lịch giao dịch và covariance history.

In [ ]:
# Cell 1 — mount Drive và clone code
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, sys, shutil, zipfile, json, os

REPO_URL = 'https://github.com/maiphuowng205/kltn.git'
REPO = Path('/content/kltn')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
print('Repo:', REPO)
print('Commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# Cell 2 — tìm và giải nén package ASEAN từ Drive
DRIVE_ROOT = Path('/content/drive/MyDrive')
LOCAL_ROOT = Path('/content/asean_v1_country_runs')

# Có thể upload dạng folder hoặc ZIP. Notebook tự tìm theo tên.
folder_candidates = [p for p in DRIVE_ROOT.rglob('asean_v1_country_runs') if p.is_dir()]
zip_candidates = [p for p in DRIVE_ROOT.rglob('*asean_v1_country_runs*.zip') if p.is_file()]
print('Folder candidates:', folder_candidates[:5])
print('ZIP candidates:', zip_candidates[:5])

if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)
if folder_candidates:
    shutil.copytree(folder_candidates[0], LOCAL_ROOT)
elif zip_candidates:
    with zipfile.ZipFile(zip_candidates[0]) as z:
        z.extractall('/content')
    extracted = Path('/content/asean_v1_country_runs')
    if not extracted.exists():
        raise FileNotFoundError('ZIP phải chứa thư mục asean_v1_country_runs ở cấp cao nhất.')
else:
    raise FileNotFoundError('Hãy upload folder asean_v1_country_runs hoặc asean_v1_country_runs.zip vào MyDrive trước khi chạy cell này.')

COUNTRIES = ['indonesia', 'malaysia', 'philippines', 'singapore', 'thailand']
for country in COUNTRIES:
    root = LOCAL_ROOT / country / 'data' / 'lseg_v3'
    print(country, root.exists(), root)

In [ ]:
# Cell 3 — chọn phạm vi chạy
# Lần đầu nên thử ['indonesia']; khi chạy ổn đổi thành COUNTRIES.
SELECTED_COUNTRIES = ['indonesia']
EPOCHS = 100
SEED = 7
BATCH_DATES = 16
RUN_FORECAST_BASELINES = True
RUN_PTCST = True
RUN_DEEP_BASELINES = True
RUN_RISK_PORTFOLIO = True
RUN_ABLATIONS = True
RUN_COST_AUDIT = True

def run(cmd):
    print('\n$', ' '.join(map(str, cmd)))
    return subprocess.run(cmd, cwd=REPO, check=True)

for country in SELECTED_COUNTRIES:
    data_root = LOCAL_ROOT / country / 'data' / 'lseg_v3'
    run([sys.executable, str(REPO/'scripts'/'validate_v3_contract.py'), '--workspace-root', str(LOCAL_ROOT/country)])
print('Preflight passed.')

In [ ]:
# Cell 4 — forecast baselines
for country in SELECTED_COUNTRIES:
    data_root = LOCAL_ROOT / country / 'data' / 'lseg_v3'
    run_dir = LOCAL_ROOT / country / 'runs' / 'v3_forecast_baselines'
    if RUN_FORECAST_BASELINES and not (run_dir/'metrics.json').exists():
        run([sys.executable, str(REPO/'scripts'/'run_v3_forecast_baselines.py'), '--data-root', str(data_root), '--run-dir', str(run_dir), '--xgb-estimators', '200'])
    print(country, 'forecast metrics:', (run_dir/'forecast_metrics_summary.parquet').exists())

In [ ]:
# Cell 5 — PTCST và deep baselines
for country in SELECTED_COUNTRIES:
    data_root = LOCAL_ROOT / country / 'data' / 'lseg_v3'
    ptcst_dir = LOCAL_ROOT / country / 'runs' / 'v3_ptcst_ca_mvo'
    if RUN_PTCST and not (ptcst_dir/'metrics.json').exists():
        run([sys.executable, str(REPO/'scripts'/'run_v3_ptcst_method.py'), '--data-root', str(data_root), '--run-dir', str(ptcst_dir), '--model-type', 'PTCST', '--epochs', str(EPOCHS), '--seed', str(SEED), '--batch-dates', str(BATCH_DATES)])
    deep_root = LOCAL_ROOT / country / 'runs' / 'v3_deep_baselines'
    if RUN_DEEP_BASELINES and not ((deep_root/'TemporalTransformer'/'metrics.json').exists() and (deep_root/'PatchTST'/'metrics.json').exists()):
        run([sys.executable, str(REPO/'scripts'/'run_v3_deep_baseline_sweep.py'), '--data-root', str(data_root), '--run-root', str(deep_root), '--epochs', str(EPOCHS)])
    print(country, 'PTCST:', (ptcst_dir/'metrics.json').exists(), 'deep:', (deep_root/'PatchTST'/'metrics.json').exists())

In [ ]:
# Cell 6 — risk, benchmark danh mục và ablation
for country in SELECTED_COUNTRIES:
    data_root = LOCAL_ROOT / country / 'data' / 'lseg_v3'
    root = LOCAL_ROOT / country / 'runs'
    if RUN_RISK_PORTFOLIO:
        risk_dir = root/'v3_risk_coverage'
        if not (risk_dir/'metrics.json').exists():
            run([sys.executable, str(REPO/'scripts'/'run_v3_risk_coverage.py'), '--data-root', str(data_root), '--run-dir', str(risk_dir)])
        portfolio_dir = root/'v3_portfolio_benchmarks'
        if not (portfolio_dir/'portfolio_metrics_summary.parquet').exists():
            run([sys.executable, str(REPO/'scripts'/'run_v3_portfolio_benchmarks.py'), '--data-root', str(data_root), '--forecast-run', str(root/'v3_forecast_baselines'), '--run-dir', str(portfolio_dir)])
    if RUN_ABLATIONS:
        abl_dir = root/'v3_ptcst_ablations'
        if not (abl_dir/'portfolio_metrics_summary.parquet').exists():
            run([sys.executable, str(REPO/'scripts'/'run_v3_ptcst_ablations.py'), '--data-root', str(data_root), '--forecast-run', str(root/'v3_ptcst_ca_mvo'), '--run-dir', str(abl_dir)])
print('Risk/portfolio stage complete.')

In [ ]:
# Cell 7 — BID/ASK cost audit và bảng tổng hợp
if RUN_COST_AUDIT:
    run([sys.executable, str(REPO/'scripts'/'run_asean_quote_cost_audit.py'), '--country-root', str(LOCAL_ROOT), '--countries', *SELECTED_COUNTRIES])
aggregate_dir = Path('/content/asean_v1_aggregate')
run([sys.executable, str(REPO/'scripts'/'aggregate_asean_runs.py'), '--country-root', str(LOCAL_ROOT), '--output-dir', str(aggregate_dir)])
print('Aggregate:', aggregate_dir)

In [ ]:
# Cell 8 — xem kết quả và đồng bộ về Drive
import pandas as pd
for name in ['deep_model_summary.csv', 'portfolio_benchmarks.csv', 'ptcst_ablations.csv', 'risk_coverage_summary.csv', 'quote_cost_summary.csv']:
    p = aggregate_dir / name
    if p.exists():
        print('\n===', name, '===')
        display(pd.read_csv(p).head(20))

DRIVE_OUTPUT = Path('/content/drive/MyDrive/kltn/asean_v1_colab_results')
if DRIVE_OUTPUT.exists():
    shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(aggregate_dir, DRIVE_OUTPUT)
for country in SELECTED_COUNTRIES:
    src = LOCAL_ROOT / country / 'runs'
    dst = DRIVE_OUTPUT / 'country_runs' / country / 'runs'
    shutil.copytree(src, dst)
print('Đã lưu kết quả vào:', DRIVE_OUTPUT)